# Imports

In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import pandas as pd
import numpy as np
import requests
from pathlib import *
import os
import sys
from dotenv import load_dotenv
from scipy.stats import pearsonr
import polars as pl
from tqdm import tqdm
from typing import Union

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)


from utils import model_connector
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage, AIMessage, SystemMessage


# Data preprocess

In [43]:
# Step 1: Load Data
def data_loader(file_path: str) -> pd.DataFrame:
    data = pd.read_csv(file_path)
    data = data.rename(columns={"Date": "date", "Open": "open", "Close": "close", "High": "high", "Low": "low" })
    data.set_index("date", inplace=True)
    return data

# Step 1.1 - Resample to lower timeframe, Ignore If Daily
def data_resampler(df: pd.DataFrame, resample_timeframe: str) -> pd.DataFrame:
    df = df.resample("15min").agg({
        "open":            "first",
        "high":            "max",
        "low":             "min",
        "close":           "last",
        "volume":          "sum",
        "quote_volume":    "sum",
        "trades":          "sum",
        "taker_buy_base":  "sum",
        "taker_buy_quote": "sum",
    }).dropna()
    return df

# Step 1.2 - We split into train and test. Since we use windowing and former values to calculate the RV and returns we don't want leakage from 
# the train data into the test data. Best approach is to split the dataset first, then calculate the features, THEN drop the na's.
def get_train_test_df(data: pd.DataFrame, split_ratio: float = 0.6) -> Union[pd.DataFrame, pd.DataFrame]:
    train_data = data[:int(len(data)*0.6)]
    test_data = data[len(train_data): ]
    return train_data, test_data

# Write Functions to create features for both train and test. 
# Step 2: Calculate Log Returns
def log_returns(data: pd.DataFrame) -> pd.DataFrame:
    data["r"] = (data["close"]/data["close"].shift(1)) - 1 
    data["log_r"] = np.log(data["close"]/data["close"].shift(1))
    return data

# Step 3: Calculate Volatility (Square of log returns)
def volatility(data: pd.DataFrame) -> pd.DataFrame:
    data["rv"] = data["log_r"]**2
    return data

def rescale(data: pd.DataFrame) -> pd.DataFrame:
    data["log_r_scaled"] = data["log_r"] * 1e3
    data["rv_scaled"] = np.log(data["rv"])
    return data

# Step 4: Collect X and Y
def features(data: pd.DataFrame, window: int, strides: int, require_rescale: bool = False) -> Union[list, list]:
    if require_rescale:
        data = data.pipe(rescale)
        sample_data = data[["log_r_scaled", "rv_scaled"]].dropna().to_numpy()
    else:
        sample_data = data[["log_r", "rv"]].dropna().to_numpy()

    w = window 
    x = [] 
    y = []

    for i in tqdm(range(0, len(sample_data)-w, strides)):
        x.append(tuple(sample_data[i:i+w]))
        
    y = [sample_data[i+w][1] for i in range(0, len(sample_data)-w, strides)]

    return x, y

In [26]:
train_data, test_data = data_loader(file_path="../data/sp500.csv").pipe(get_train_test_df, split_ratio=0.6)

In [44]:
window = 7
strides = 1

# Train Prepare
x_train, y_train = train_data.pipe(log_returns).pipe(volatility).pipe(features, window=window, strides=strides)
# Test Prepare
x_test, y_test = test_data.pipe(log_returns).pipe(volatility).pipe(features, window=window, strides=strides)

print(f"Len of x_train: {len(x_train)}, Len of y_train: {len(y_train)}")
print(f"Len of x_test: {len(x_test)}, Len of y_test: {len(y_test)}")

100%|██████████| 1363/1363 [00:00<00:00, 143071.13it/s]

Len of x_train: 2048, Len of y_train: 2048
Len of x_test: 1363, Len of y_test: 1363


# Connect model

In [45]:
def formulate_p_input(x: tuple, p_input_prompt: str) -> HumanMessage:
    return HumanMessage(f"{p_input_prompt}\n{x}")
    
def formulate_p_query(p_query_prompt: str) -> SystemMessage:
    return SystemMessage(p_query_prompt)

In [ ]:
p_input_prompt = """You are given a chronological sequence of observations
for a financial asset.
Each row reports the 1-period log return log_r and the realized variance rv.
where
rv = log_r^2.
"""

p_query_prompt = """Task: forecast rv at the next time step, t+1.

The forecast target is one step ahead of the final observation above,
reported in the same units as the sequence.

Respond with a single number. Output nothing
else — no explanation, no units, no surrounding text.

rv =

If Unable to generate forecast return NA.
"""


def get_initial_predictions(model, p_query_prompt: str, p_input_prompt: str):
    for i in tqdm(range(len(x_train[:10]))):
        # if i == 5:
        #     break
        p_input = formulate_p_input(x_train[i], p_input_prompt = p_input_prompt)
        p_query = formulate_p_query(p_query_prompt=p_query_prompt)
        p_conc = p_input
        response = model.invoke([p_query, p_input])
        yield p_conc, float(response.content)

In [49]:
model: ChatOpenAI = model_connector.get_model(model_name="OpenAI")
# model: ChatOpenAI = model_connector.get_model(model_name=None)
initial_predictions = []
with open('outputs_1.csv', 'w') as model_output:
    model_output.write("pconc,initial_prediction\n")
    for pconc, pred in get_initial_predictions(model, p_query_prompt=p_query_prompt, p_input_prompt=p_input_prompt):
        # initial_predictions.append(pred)
        model_output.write(f"{pconc},{pred}\n")

  0%|          | 0/10 [00:00<?, ?it/s]


NameError: name 'pconc' is not defined